# I. Librerías y configuración inicial

In [3]:
import pandas as pd
import pyarrow.parquet as pq

* Parámetros y constantes

* Rutas de acceso

In [2]:
ruta = "../data/processed/transacciones_bogota_cundinamarca.parquet"

## II. Carga y exploración inicial del archivo

## Exploración de tabla

In [6]:
archivo = pq.ParquetFile(ruta)
archivo.schema.names

['PK',
 'MATRICULA',
 'FECHA_RADICA_TEXTO',
 'FECHA_APERTURA_TEXTO',
 'YEAR_RADICA',
 'ORIP',
 'DIVIPOLA',
 'DEPARTAMENTO',
 'MUNICIPIO',
 'TIPO_PREDIO_ZONA',
 'CATEGORIA_RURALIDAD',
 'NUM_ANOTACION',
 'ESTADO_FOLIO',
 'FOLIOS_DERIVADOS',
 'Dinámica_Inmobiliaria',
 'COD_NATUJUR',
 'NOMBRE_NATUJUR',
 'NUMERO_CATASTRAL',
 'NUMERO_CATASTRAL_ANTIGUO',
 'DOCUMENTO_JUSTIFICATIVO',
 'COUNT_A',
 'COUNT_DE',
 'PREDIOS_NUEVOS',
 'TIENE_VALOR',
 'TIENE_MAS_DE_UN_VALOR',
 'VALOR']

In [7]:
print("Número de filas:", archivo.metadata.num_rows)
print("Número de columnas:", archivo.metadata.num_columns)
print("Número de grupos:", archivo.metadata.num_row_groups)

Número de filas: 7008966
Número de columnas: 26
Número de grupos: 33


In [11]:
columnas_objetivo = [
    "VALOR",
    "TIENE_VALOR",
    "TIENE_MAS_DE_UN_VALOR",
    "COD_NATUJUR",
    "NOMBRE_NATUJUR",
    "YEAR_RADICA"
]

df_objetivo = pd.read_parquet(ruta,columns=columnas_objetivo)
display(df_objetivo.head())
df_objetivo.info()

,VALOR,TIENE_VALOR,TIENE_MAS_DE_UN_VALOR,COD_NATUJUR,NOMBRE_NATUJUR,YEAR_RADICA
0,"42,000,000",1,0,125,COMPRAVENTA,2015
1,0,1,0,901,ACLARACIÓN,2015
2,"55,000,000",1,0,125,COMPRAVENTA,2022
3,<NA>,0,0,901,ACLARACIÓN,2018
4,<NA>,0,0,901,ACLARACIÓN,2018


<class 'pandas.DataFrame'>
RangeIndex: 7008966 entries, 0 to 7008965
Data columns (total 6 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   VALOR                  string
 1   TIENE_VALOR            Int64 
 2   TIENE_MAS_DE_UN_VALOR  Int64 
 3   COD_NATUJUR            string
 4   NOMBRE_NATUJUR         string
 5   YEAR_RADICA            Int64 
dtypes: Int64(3), string(3)
memory usage: 598.9 MB


## Transformaciones pre modelación

Durante esta sesión se inició la reformulación del proyecto para ajustarlo a un enfoque de aprendizaje supervisado. Con el fin de mantener la trazabilidad de los cambios, primero se actualizó la rama principal del repositorio y posteriormente se creó una nueva rama de trabajo llamada `redefinicion-supervisado`. A partir de allí se creó un notebook específico para documentar el proceso de análisis y las decisiones relacionadas con la nueva formulación del problema.

En el notebook se revisó la estructura del archivo Parquet versionado mediante DVC y se identificaron las variables disponibles. Se evaluaron varias posibles variables objetivo y se seleccionó `VALOR` como la principal candidata para formular un problema de regresión. Antes de modelar, se realizó una revisión de calidad sobre esta variable, transformándola de texto a formato numérico y verificando que no existieran errores de conversión.

El análisis permitió identificar tres grupos principales: registros con valor faltante, registros con valor igual a cero y registros con valor positivo. Posteriormente, se comparó esta información con la variable `TIENE_VALOR` y se observó que no todos los registros marcados como transacciones con valor contienen necesariamente un monto positivo, por lo que no es conveniente utilizar directamente esta variable como filtro para construir el conjunto de entrenamiento.

También se analizó la distribución de `VALOR` según el tipo de acto jurídico. Los resultados mostraron que `COMPRAVENTA` concentra la mayor cantidad de registros con valor monetario positivo, con una diferencia amplia frente a otros actos jurídicos. A partir de este resultado se planteó como alternativa preliminar redefinir el problema como la estimación del valor monetario de compraventas inmobiliarias utilizando la información disponible de cada transacción.


SEGUIR COMPLETANDO A MEDIDA QUE SE AVANCE

Como siguiente paso queda pendiente revisar la variable `TIENE_MAS_DE_UN_VALOR` dentro de las compraventas y posteriormente analizar qué variables explicativas pueden utilizarse para construir el modelo supervisado, evitando posibles problemas de fuga de información.

In [17]:
# Paso 1 --> Convertir la columna Valor a numérica para poderla usar como problema supervisado
valor_limpio = (df_objetivo["VALOR"].str.replace(",", "", regex=False).str.strip())
df_objetivo["VALOR_NUM"] = pd.to_numeric(valor_limpio,errors="coerce")
display(df_objetivo[["VALOR", "VALOR_NUM"]].head())

conversion_fallida = (df_objetivo["VALOR"].notna() & df_objetivo["VALOR_NUM"].isna())
print("Valores no nulos que no se pudieron convertir:", conversion_fallida.sum())

print("Total registros:", len(df_objetivo))
print("VALOR faltante:", df_objetivo["VALOR_NUM"].isna().sum())
print("VALOR = 0:", (df_objetivo["VALOR_NUM"] == 0).sum())
print("VALOR > 0:", (df_objetivo["VALOR_NUM"] > 0).sum())
print("VALOR < 0:", (df_objetivo["VALOR_NUM"] < 0).sum())

pd.crosstab(
    df_objetivo["TIENE_VALOR"],
    [
        df_objetivo["VALOR_NUM"].isna(),
        df_objetivo["VALOR_NUM"].eq(0)
    ],
    rownames=["TIENE_VALOR"],
    colnames=["Es nulo", "Es cero"]
)

,VALOR,VALOR_NUM
0,"42,000,000",42000000.0
1,0,0.0
2,"55,000,000",55000000.0
3,<NA>,<NA>
4,<NA>,<NA>


Valores no nulos que no se pudieron convertir: 0
Total registros: 7008966
VALOR faltante: 5112656
VALOR = 0: 806588
VALOR > 0: 1089722
VALOR < 0: 0


Es nulo        False        
Es cero        False   True 
TIENE_VALOR                 
1            1089722  806588

In [ ]:
df_objetivo["ESTADO_VALOR"] = "Faltante"
df_objetivo.loc[df_objetivo["VALOR_NUM"].eq(0),"ESTADO_VALOR"] = "Cero"
df_objetivo.loc[df_objetivo["VALOR_NUM"].gt(0),"ESTADO_VALOR"] = "Positivo"

tabla_actos = pd.crosstab(df_objetivo["NOMBRE_NATUJUR"],df_objetivo["ESTADO_VALOR"])
tabla_actos.sort_values("Positivo",ascending=False).head(20)


ESTADO_VALOR,Cero,Faltante,Positivo
NOMBRE_NATUJUR,,,
COMPRAVENTA,1369,564746,608372
CANCELACIÓN POR VOLUNTAD DE LAS PARTES,49343,708912,182526
COMPRAVENTA DERECHOS DE CUOTA,571,66552,75328
CANCELACIÓN PARCIAL,630,35905,59969
ADJUDICACIÓN EN SUCESION,4021,72278,21088
TRANSFERENCIA DE DOMINIO A TÍTULO DE BENEFICIO EN FIDUCIA MERCANTIL,70,53699,15788
DECLARACIÓN DE CONSTRUCCIÓN EN SUELO PROPIO,420,13401,14013
HIPOTECA ABIERTA SIN LIMITE DE CUANTIA,89688,485987,13207
COMPRAVENTA DERECHOS Y ACCIONES,109,10577,10312
